In [1]:
from rag_helper import RAGBase
from ingest import load_faq_data,build_index
from google import genai
from dotenv import load_dotenv
import json

In [2]:
load_dotenv()
geminiClient = genai.Client()

documents = load_faq_data()
index = build_index(documents)

In [3]:
system_instructions = f"""
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Do not make up answers. If you don't know the answer, say "I don't know" and suggest that the user contact the course administrator for more information.

At the end, ask if there are other areas that the user wants to explore.
"""
user_prompt = 'can you explain about all the courses available?'

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


In [ ]:
def agent_loop(instructions, user_prompt):
    print("Starting agent loop...")
    iteration = 1
    current_user_promopt = user_prompt
    previous_reponse_id=None

    while True:
        has_function_call = False
        initial_response = geminiClient.interactions.create(
            model='gemini-3.5-flash-lite',
            input=current_user_promopt,
            system_instruction=instructions,
            previous_interaction_id=previous_reponse_id,
            tools=[search_tool]
        )
        print(f"Iteration {iteration}: Initial response received.")
        print("intial Response:", initial_response)

        for step in initial_response.steps:
            if step.type == 'function_call':
                call_argument = make_call(step)
                has_function_call = True

            elif step.type == 'message':
                final_answer = step.output_text
                has_function_call = False

        iteration += 1
        
        if not has_function_call:
            current_user_promopt = call_argument
            previous_reponse_id = initial_response.id
            break

    return print("Final Answer \n:", final_answer)

In [ ]:
def make_call(step):
    print(f"Function call detected: {step.name} with arguments {step.arguments}")
    call_argument=step

    if(step.name == 'search'):
        search_results = search(call_argument.arguments['query'])
        print('Search Results:', search_results)

    result_json = json.dumps(search_results, indent=2)
    return result_json        

In [8]:
agent_loop(system_instructions, user_prompt)

Starting agent loop...
Iteration 1: Initial response received.
intial Response: status='requires_action' agent=None agent_config=None cached_content=None created='2026-11-01T09:55:29Z' environment=None environment_id=None errors=None generation_config=None id='v1_ChdrQXpuYXBHQ01yald6N3NQMXRXa2lRVRIXa0F6bmFwR0NNcmpXejdzUDF0V2tpUVU' input=None labels=None model='gemini-3.5-flash-lite' output_audio=None output_image=None output_text='' output_video=None previous_interaction_id=None response_format=None response_mime_type=None response_modalities=None safety_settings=None service_tier='standard' steps=[ThoughtStep(signature='EmAKXgFpFH0TB85VQ/EQAx9e9CppjlwrXoisBu/iFXOdCsj4Pc5/xdILW/LzjCRhn1aFUlH96uS+hUKFifs1SKR5i94s57SU0fDhGDcjf2Q7EqdEwWYPlKkPRPXxbWk9DVY=', summary=None, type='thought'), FunctionCallStep(arguments={'query': 'courses available'}, id='call_963391', name='search', type='function_call')] system_instruction=None tools=None updated='2026-11-01T09:55:29Z' usage=Usage(cached_token

KeyboardInterrupt: 